System construction and test


In [2]:
from datetime import date, datetime
import pandas as pd
import time
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'
from itertools import product
import sqlite3
from Stoch_HighMedLow_Long import *
from stoploss import *
from index import *
from metrics import *
from backtest import *
from concurrent.futures import ThreadPoolExecutor
import traceback

In [3]:
#seleçonar backtest :  1	1	2020-04-03 19:30:00	2024-04-03 19:30:00	GGAL	60min	USD	C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db
# no DB sqTradeSys
id_backtest = 1

In [4]:
srbacktest = backtest_parameters (id_backtest)
display (srbacktest)

id_backtest                                                    1
id_titulos                                                     1
dataini                                      2020-04-03 19:30:00
datafim                                      2024-04-03 19:30:00
symbol                                                      GGAL
intervalo                                                  60min
moeda                                                        USD
source         C:\Users\scitr\anaconda_projects\Trading_Syste...
Name: 0, dtype: object

In [5]:

dftitulosdados = titulos_dados (srbacktest)
display(len(dftitulosdados))
display(dftitulosdados.head(10))

8957

,datetime,open,high,low,close,volume
0,2020-04-06 08:00:00,5.8034,5.8034,5.8034,5.8034,100.0
1,2020-04-06 09:00:00,6.0432,6.2510,5.9952,6.0832,114582.0
2,2020-04-06 10:00:00,6.0945,6.0945,5.6635,5.7954,159266.0
3,2020-04-06 11:00:00,5.7954,5.8034,5.4837,5.6835,242263.0
4,2020-04-06 12:00:00,5.6715,5.8354,5.6355,5.8274,186504.0
5,2020-04-06 13:00:00,5.8274,5.8354,5.7474,5.7834,68220.0
6,2020-04-06 14:00:00,5.7714,5.7794,5.6435,5.6435,84043.0
7,2020-04-06 15:00:00,5.6595,5.6595,5.5316,5.5956,122920.0
8,2020-04-06 16:00:00,5.6036,5.6036,5.6036,5.6036,49293.0
9,2020-04-07 09:00:00,5.9793,6.0672,5.7395,6.0032,101621.0


In [9]:

dfcomb = scripts_parameters(id_backtest)
display (dfcomb)

,name,type,max,min,step
0,K,int,30,20,2
1,D,int,15,15,1
2,smoth,int,12,11,1
3,medM,int,6,6,1
4,lowM,int,5,5,1
5,stpl,float,0.02,0.02,0.01
6,comission,float,0.003,0.003,0.001
7,drawmax,float,0.2,0.02,0.1


In [19]:
                      



dfparamtest = parameters_combinator(dfcomb)
print(dfparamtest)


     K   D  smoth  medM  lowM  stpl  comission  drawmax
0   20  15     11     6     5  0.02      0.003     0.02
1   20  15     11     6     5  0.02      0.003     0.12
2   20  15     11     6     5  0.02      0.003     0.22
3   20  15     12     6     5  0.02      0.003     0.02
4   20  15     12     6     5  0.02      0.003     0.12
5   20  15     12     6     5  0.02      0.003     0.22
6   22  15     11     6     5  0.02      0.003     0.02
7   22  15     11     6     5  0.02      0.003     0.12
8   22  15     11     6     5  0.02      0.003     0.22
9   22  15     12     6     5  0.02      0.003     0.02
10  22  15     12     6     5  0.02      0.003     0.12
11  22  15     12     6     5  0.02      0.003     0.22
12  24  15     11     6     5  0.02      0.003     0.02
13  24  15     11     6     5  0.02      0.003     0.12
14  24  15     11     6     5  0.02      0.003     0.22
15  24  15     12     6     5  0.02      0.003     0.02
16  24  15     12     6     5  0.02      0.003  

In [21]:
print(len(dfparamtest)*len(dftitulosdados))

322452


START LOOP

In [26]:

dfmetricas = processar_parametros_simple (dfparamtest, dftitulosdados, srbacktest)      
display (dfmetricas)

,K,D,smoth,medM,lowM,stpl,comission,drawmax,tirtotalanual,tiranualquant,...,diasoutmedia,diasoutmax,diasoutmin,diasoutstd,stopsysfirst,stopsysquant,stopsysmedia,stopsysmaximo,stopsysminimo,stopsystd
0,20.0,15.0,11.0,6.0,5.0,0.02,0.003,0.02,0.039553,3.0,...,38.04,120.0,2.0,36.74,97.679857,20.0,103.075355,119.326036,87.379861,7.929295
1,20.0,15.0,11.0,6.0,5.0,0.02,0.003,0.12,0.039553,3.0,...,38.04,120.0,2.0,36.74,87.379861,3.0,96.135883,107.115319,87.379861,10.053843
2,20.0,15.0,11.0,6.0,5.0,0.02,0.003,0.22,0.039553,3.0,...,38.04,120.0,2.0,36.74,93.912471,1.0,93.912471,93.912471,93.912471,NaN
3,20.0,15.0,12.0,6.0,5.0,0.02,0.003,0.02,-0.081638,3.0,...,36.52,146.0,1.0,39.53,97.495797,27.0,88.991345,105.879261,71.114998,10.287505
4,20.0,15.0,12.0,6.0,5.0,0.02,0.003,0.12,-0.081638,3.0,...,36.52,146.0,1.0,39.53,94.235504,4.0,85.828708,94.235504,78.193854,8.355830
5,20.0,15.0,12.0,6.0,5.0,0.02,0.003,0.22,-0.081638,3.0,...,36.52,146.0,1.0,39.53,84.247942,1.0,84.247942,84.247942,84.247942,NaN
6,22.0,15.0,11.0,6.0,5.0,0.02,0.003,0.02,0.090441,3.0,...,44.82,145.0,1.0,42.64,97.360957,17.0,108.711620,126.487475,89.058712,9.154571
7,22.0,15.0,11.0,6.0,5.0,0.02,0.003,0.12,0.090441,3.0,...,44.82,145.0,1.0,42.64,105.623166,1.0,105.623166,105.623166,105.623166,NaN
8,22.0,15.0,11.0,6.0,5.0,0.02,0.003,0.22,0.090441,3.0,...,44.82,145.0,1.0,42.64,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
9,22.0,15.0,12.0,6.0,5.0,0.02,0.003,0.02,-0.110372,3.0,...,40.88,141.0,1.0,39.53,97.675415,27.0,85.336299,106.216865,64.117965,12.143791


In [66]:
from concurrent.futures import ThreadPoolExecutor
import traceback

def processar_parametros_multinucleo(dfparamtest, dftitulosdados, srbacktest, dfmetricas):

    dfmetricas = None

    def processar_parametros(args):
        il, linha, dftitulosdados, dataini, datafim = args
        try:
            K, D, smoth, medM, lowM, stpl, comission, drawmax, lsmetricas = parametros(dfparamtest, il)

            dfsignals = Stoch_HighMedLow_Long(dftitulosdados, K, D, smoth, medM, lowM)
            dfsignals, dfstoploss = stop_loss_reentry(dfsignals, stpl)
            dfindex = index_calculation(dfsignals, comission)
            dfindexdrawdown = stop_drawdown_simple(dfindex, drawmax)

            dfinputmetricas = dfindex[['datetime', 'state', 'index_sc', 'index', 'trade']]
            setirtotalanual, lsmetricas = tir_total_anualizada(dfinputmetricas, lsmetricas)
            dftiranual = tir_anuais_df(dfinputmetricas, dataini, datafim)
            setiranuaisestats, lsmetricas = tir_anuais_estats(dftiranual, lsmetricas)
            setradesestats, lsmetricas = trades_estats(dfinputmetricas, lsmetricas)
            dfdrawdowns = drawdowns_df(dfinputmetricas)
            sedrawdownsestats, lsmetricas = drawdowns_estats(dfdrawdowns, lsmetricas)
            dfdiasout = dias_out_df(dfinputmetricas)
            sediasoutestats, lsmetricas = dias_out_estats(dfdiasout, lsmetricas)
            dfstopdrawdown = stop_drawdown_df(dfindexdrawdown)
            sestopdrawdownestats, lsmetricas = stop_drawdown_estats(dfstopdrawdown, lsmetricas)

            return lsmetricas

        except Exception as e:
            print(f"⚠️ Erro ao processar índice {il}: {e}")
            traceback.print_exc()
            return None

    # Prepara os argumentos para cada linha
    args_list = [
        (il, dfparamtest.iloc[il], dftitulosdados, srbacktest['dataini'], srbacktest['datafim'])
        for il in dfparamtest.index
    ]

    # Executa em paralelo
    resultados = []
    with ThreadPoolExecutor() as executor:
        for resultado in executor.map(processar_parametros, args_list):
            if resultado is not None:
                resultados.append(resultado)

    # Atualiza dfmetricas com os resultados válidos
    for lsmetricas in resultados:
        dfmetricas = atualizar_df_metricas(dfmetricas, lsmetricas)

    return dfmetricas

In [68]:
dfmetricas = processar_parametros_multinucleo(dfparamtest, dftitulosdados, srbacktest, dfmetricas)

END LOOP

In [70]:
dfmetricas.insert(
    loc=0,  # insere como primeira coluna
    column="id_backtest",
    value=[srbacktest["id_backtest"]] * len(dfmetricas)
)
display(dfmetricas)

,id_backtest,K,D,smoth,medM,lowM,stpl,comission,drawmax,tirtotalanual,...,diasoutmedia,diasoutmax,diasoutmin,diasoutstd,stopsysfirst,stopsysquant,stopsysmedia,stopsysmaximo,stopsysminimo,stopsystd
0,1,20.0,15.0,11.0,6.0,5.0,0.02,0.003,0.02,0.039553,...,38.04,120.0,2.0,36.74,97.679857,20.0,103.075355,119.326036,87.379861,7.929295
1,1,20.0,15.0,11.0,6.0,5.0,0.02,0.003,0.12,0.039553,...,38.04,120.0,2.0,36.74,87.379861,3.0,96.135883,107.115319,87.379861,10.053843
2,1,20.0,15.0,11.0,6.0,5.0,0.02,0.003,0.22,0.039553,...,38.04,120.0,2.0,36.74,93.912471,1.0,93.912471,93.912471,93.912471,NaN
3,1,20.0,15.0,12.0,6.0,5.0,0.02,0.003,0.02,-0.081638,...,36.52,146.0,1.0,39.53,97.495797,27.0,88.991345,105.879261,71.114998,10.287505
4,1,20.0,15.0,12.0,6.0,5.0,0.02,0.003,0.12,-0.081638,...,36.52,146.0,1.0,39.53,94.235504,4.0,85.828708,94.235504,78.193854,8.355830
5,1,20.0,15.0,12.0,6.0,5.0,0.02,0.003,0.22,-0.081638,...,36.52,146.0,1.0,39.53,84.247942,1.0,84.247942,84.247942,84.247942,NaN
6,1,22.0,15.0,11.0,6.0,5.0,0.02,0.003,0.02,0.090441,...,44.82,145.0,1.0,42.64,97.360957,17.0,108.711620,126.487475,89.058712,9.154571
7,1,22.0,15.0,11.0,6.0,5.0,0.02,0.003,0.12,0.090441,...,44.82,145.0,1.0,42.64,105.623166,1.0,105.623166,105.623166,105.623166,NaN
8,1,22.0,15.0,11.0,6.0,5.0,0.02,0.003,0.22,0.090441,...,44.82,145.0,1.0,42.64,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
9,1,22.0,15.0,12.0,6.0,5.0,0.02,0.003,0.02,-0.110372,...,40.88,141.0,1.0,39.53,97.675415,27.0,85.336299,106.216865,64.117965,12.143791
